# Petri_curcle: split, train (`yolo26n.pt`), test

Notebook pipeline:
1. Prepare config and paths
2. Split dataset into `train/val/test`
3. Train YOLO model from `yolo26n.pt`
4. Evaluate on test split
5. Plot training curves and evaluation artifacts


In [ ]:
# Uncomment if required
# %pip install -q ultralytics pyyaml pandas matplotlib pillow

from pathlib import Path
import random
import shutil
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import yaml
from ultralytics import YOLO
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "Petri_curcle"

SOURCE_IMAGES = DATASET_ROOT / "images" / "train"
SOURCE_LABELS = DATASET_ROOT / "labels" / "train"

SPLIT_ROOT = DATASET_ROOT / "split_dataset"
SPLITS = {"train": 0.7, "val": 0.2, "test": 0.1}
assert abs(sum(SPLITS.values()) - 1.0) < 1e-9

MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = None  # None=auto, "cpu" for CPU

TRAIN_PROJECT = "runs/petri_curcle"
TRAIN_NAME = "yolo26n_petri_curcle"

# Offline soft augmentation (files are physically added to split_dataset)
USE_OFFLINE_SOFT_AUG = True
AUG_TARGET_SPLITS = ["train"]
AUG_MULTIPLIER = 5  # total factor including originals
AUG_CLEAR_PREVIOUS = True

# Online soft augmentation (Ultralytics train args)
YOLO_TRAIN_AUG_ARGS = {
    "hsv_h": 0.01,
    "hsv_s": 0.22,
    "hsv_v": 0.14,
    "degrees": 3.0,
    "translate": 0.04,
    "scale": 0.08,
    "shear": 1.0,
    "perspective": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "erasing": 0.1,
}

print(f"Dataset root: {DATASET_ROOT.resolve()}")
print(f"Source images: {SOURCE_IMAGES}")
print(f"Source labels: {SOURCE_LABELS}")
print(f"Offline aug enabled: {USE_OFFLINE_SOFT_AUG}, multiplier={AUG_MULTIPLIER}, splits={AUG_TARGET_SPLITS}")



In [ ]:
# Split source train set into train/val/test and create data_split.yaml
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
image_files = sorted([p for p in SOURCE_IMAGES.glob("*") if p.suffix.lower() in image_exts])

pairs = []
missing_labels = []
for img in image_files:
    lbl = SOURCE_LABELS / f"{img.stem}.txt"
    if lbl.exists():
        pairs.append((img, lbl))
    else:
        missing_labels.append(img.name)

if not pairs:
    raise RuntimeError("No image-label pairs found in source directories.")

idx = list(range(len(pairs)))
rng = random.Random(SEED)
rng.shuffle(idx)

n_total = len(idx)
n_train = int(n_total * SPLITS["train"])
n_val = int(n_total * SPLITS["val"])
n_test = n_total - n_train - n_val

split_indices = {
    "train": idx[:n_train],
    "val": idx[n_train:n_train + n_val],
    "test": idx[n_train + n_val:],
}

if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

for split in split_indices:
    (SPLIT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

for split, indices in split_indices.items():
    for i in indices:
        img, lbl = pairs[i]
        shutil.copy2(img, SPLIT_ROOT / "images" / split / img.name)
        shutil.copy2(lbl, SPLIT_ROOT / "labels" / split / lbl.name)

data_yaml = {
    "path": str(SPLIT_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "Petri_curcle"},
}

data_yaml_path = SPLIT_ROOT / "data_split.yaml"
with data_yaml_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

summary_df = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "images": [len(split_indices["train"]), len(split_indices["val"]), len(split_indices["test"])],
    }
)
summary_df["labels"] = summary_df["images"]

if missing_labels:
    print(f"Missing labels: {len(missing_labels)} files (ignored)")

print(f"Total valid pairs: {len(pairs)}")
print(f"Split root: {SPLIT_ROOT.resolve()}")
print(f"data yaml: {data_yaml_path}")
display(summary_df)


In [ ]:
# Soft offline augmentation for Petri_curcle/split_dataset
# Creates extra files like IMG_1234__soft01.jpg + IMG_1234__soft01.txt
import cv2
import re

soft_suffix_re = re.compile(r"__soft\d{2}$")
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def gamma_correct(image, gamma):
    lut = np.array([((x / 255.0) ** gamma) * 255.0 for x in range(256)], dtype=np.float32)
    lut = np.clip(lut, 0, 255).astype(np.uint8)
    return cv2.LUT(image, lut)


def mild_color_jitter(image, rng):
    out = image.astype(np.float32)
    alpha = float(rng.uniform(0.95, 1.07))
    beta = float(rng.uniform(-8.0, 8.0))
    out = np.clip(out * alpha + beta, 0, 255).astype(np.uint8)
    out = gamma_correct(out, gamma=float(rng.uniform(0.93, 1.07)))

    hsv = cv2.cvtColor(out, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 1] *= float(rng.uniform(0.93, 1.08))
    hsv[..., 2] *= float(rng.uniform(0.95, 1.05))
    return cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)


def mild_noise_blur(image, rng):
    sigma_noise = float(rng.uniform(2.0, 5.0))
    noisy = image.astype(np.float32) + rng.normal(0.0, sigma_noise, size=image.shape).astype(np.float32)
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    k = 3 if float(rng.random()) < 0.75 else 5
    sigma_blur = float(rng.uniform(0.2, 0.9))
    return cv2.GaussianBlur(noisy, (k, k), sigmaX=sigma_blur)


def mild_clahe(image, rng):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=float(rng.uniform(1.2, 1.8)), tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    out = cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)
    return mild_color_jitter(out, rng)


def mild_sharpen(image, rng):
    blur = cv2.GaussianBlur(image, (0, 0), sigmaX=float(rng.uniform(0.4, 0.8)))
    return cv2.addWeighted(image, 1.12, blur, -0.12, 0.0)


def soft_augment(image, variant_idx, rng):
    mode = (variant_idx - 1) % 4
    if mode == 0:
        return mild_color_jitter(image, rng)
    if mode == 1:
        return mild_noise_blur(image, rng)
    if mode == 2:
        return mild_clahe(image, rng)
    return mild_sharpen(mild_color_jitter(image, rng), rng)


def collect_pairs(images_dir, labels_dir):
    pairs = []
    for img_path in sorted(images_dir.glob("*")):
        if not img_path.is_file() or img_path.suffix.lower() not in image_exts:
            continue
        if soft_suffix_re.search(img_path.stem):
            continue
        lbl_path = labels_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            pairs.append((img_path, lbl_path))
    return pairs


def clear_previous_soft(images_dir, labels_dir):
    removed = 0
    for p in images_dir.glob("*"):
        if p.is_file() and soft_suffix_re.search(p.stem):
            p.unlink()
            removed += 1
    for p in labels_dir.glob("*.txt"):
        if p.is_file() and soft_suffix_re.search(p.stem):
            p.unlink()
            removed += 1
    return removed


aug_report = {
    "dataset_root": str(SPLIT_ROOT.resolve()),
    "splits": {},
    "settings": {
        "use_offline_soft_aug": bool(USE_OFFLINE_SOFT_AUG),
        "aug_target_splits": list(AUG_TARGET_SPLITS),
        "aug_multiplier": int(AUG_MULTIPLIER),
        "aug_clear_previous": bool(AUG_CLEAR_PREVIOUS),
    },
}

if not USE_OFFLINE_SOFT_AUG:
    print("Offline augmentation disabled, skipping.")
else:
    if AUG_MULTIPLIER < 2:
        raise ValueError("AUG_MULTIPLIER must be >= 2")

    rng = np.random.default_rng(SEED)
    for split in AUG_TARGET_SPLITS:
        images_dir = SPLIT_ROOT / "images" / split
        labels_dir = SPLIT_ROOT / "labels" / split
        if not images_dir.exists() or not labels_dir.exists():
            raise FileNotFoundError(f"Split not found: {split}")

        removed = clear_previous_soft(images_dir, labels_dir) if AUG_CLEAR_PREVIOUS else 0
        base_pairs = collect_pairs(images_dir, labels_dir)

        created = 0
        failed = 0
        for img_path, lbl_path in base_pairs:
            image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            if image is None:
                failed += (AUG_MULTIPLIER - 1)
                continue

            label_text = lbl_path.read_text(encoding="utf-8")
            for aug_idx in range(1, AUG_MULTIPLIER):
                out_stem = f"{img_path.stem}__soft{aug_idx:02d}"
                out_img = images_dir / f"{out_stem}{img_path.suffix.lower()}"
                out_lbl = labels_dir / f"{out_stem}.txt"

                aug_img = soft_augment(image, aug_idx, rng)
                ok = cv2.imwrite(str(out_img), aug_img)
                if not ok:
                    failed += 1
                    continue
                out_lbl.write_text(label_text, encoding="utf-8")
                created += 1

        all_img_stems = {
            p.stem for p in images_dir.glob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        }
        all_lbl_stems = {p.stem for p in labels_dir.glob("*.txt") if p.is_file()}
        matched = len(all_img_stems & all_lbl_stems)

        aug_report["splits"][split] = {
            "base_pairs": len(base_pairs),
            "created_pairs": created,
            "failed_pairs": failed,
            "removed_previous_soft_files": removed,
            "matched_pairs_after": matched,
            "expected_after": len(base_pairs) * AUG_MULTIPLIER,
        }

        print(
            f"[{split}] base={len(base_pairs)} created={created} failed={failed} "
            f"matched_after={matched} expected={len(base_pairs) * AUG_MULTIPLIER}"
        )

    aug_report["generated_at_utc"] = pd.Timestamp.utcnow().isoformat()
    aug_report_path = SPLIT_ROOT / "soft_aug_report_notebook.json"
    with aug_report_path.open("w", encoding="utf-8") as f:
        json.dump(aug_report, f, ensure_ascii=False, indent=2)
    print(f"Saved augmentation report: {aug_report_path}")



In [ ]:
# Plot split distribution
fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#3b82f6", "#f59e0b", "#10b981"]
ax.bar(summary_df["split"], summary_df["images"], color=colors)
ax.set_title("Samples per split")
ax.set_xlabel("Split")
ax.set_ylabel("Image count")
for i, v in enumerate(summary_df["images"].tolist()):
    ax.text(i, v + 0.5, str(v), ha="center")
plt.show()


In [ ]:
# Train YOLO from yolo26n.pt
# If yolo26n.pt is not local, Ultralytics may try to resolve/download it by name.
model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=TRAIN_PROJECT,
    name=TRAIN_NAME,
    exist_ok=True,
    plots=True,
    **YOLO_TRAIN_AUG_ARGS,
)

run_dir = Path(model.trainer.save_dir)
print(f"Train run dir: {run_dir}")
run_dir



In [ ]:
# Plot training curves from results.csv
results_csv = run_dir / "results.csv"
if not results_csv.exists():
    raise FileNotFoundError(f"results.csv not found: {results_csv}")

train_df = pd.read_csv(results_csv)
if "epoch" not in train_df.columns:
    train_df["epoch"] = np.arange(len(train_df))

loss_cols = [c for c in train_df.columns if "loss" in c.lower() and "lr" not in c.lower()]
metric_cols = [
    c for c in train_df.columns
    if any(k in c.lower() for k in ["precision", "recall", "map50", "map"]) and "class" not in c.lower()
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if loss_cols:
    for col in loss_cols:
        axes[0].plot(train_df["epoch"], train_df[col], label=col)
axes[0].set_title("Loss curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(loc="best", fontsize=8)

if metric_cols:
    for col in metric_cols:
        axes[1].plot(train_df["epoch"], train_df[col], label=col)
axes[1].set_title("Metrics curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Metric")
axes[1].legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate best checkpoint on test split
best_weights = run_dir / "weights" / "best.pt"
if not best_weights.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {best_weights}")

best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(data_yaml_path),
    split="test",
    project=TRAIN_PROJECT,
    name=f"{TRAIN_NAME}_test",
    exist_ok=True,
    plots=True,
    save_json=True,
)

def to_float(v):
    try:
        return float(v)
    except Exception:
        return float("nan")

metrics_summary = {
    "precision_B": to_float(test_metrics.box.mp),
    "recall_B": to_float(test_metrics.box.mr),
    "mAP50_B": to_float(test_metrics.box.map50),
    "mAP50_95_B": to_float(test_metrics.box.map),
    "fitness": to_float(getattr(test_metrics, "fitness", float("nan"))),
}

test_dir = Path(test_metrics.save_dir)
metrics_df = pd.DataFrame({"value": metrics_summary})
display(metrics_df)

metrics_json_path = test_dir / "test_metrics_summary.json"
with metrics_json_path.open("w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

print(f"Test run dir: {test_dir}")
print(f"Saved metrics JSON: {metrics_json_path}")


In [ ]:
# Show evaluation plots (PR/F1/Confusion Matrix) if present
plot_candidates = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
]

plot_paths = [test_dir / p for p in plot_candidates if (test_dir / p).exists()]
if not plot_paths:
    print("No evaluation plot images found in test run directory.")
else:
    ncols = 2
    nrows = int(np.ceil(len(plot_paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, p in zip(axes, plot_paths):
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(p.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# Show sample predictions on test images
sample_images = sorted((SPLIT_ROOT / "images" / "test").glob("*"))
if not sample_images:
    print("No test images found for prediction preview.")
else:
    sample_images = sample_images[:6]
    preds = best_model.predict(
        source=[str(p) for p in sample_images],
        conf=0.25,
        iou=0.5,
        verbose=False,
    )

    n = len(sample_images)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, pred, src in zip(axes, preds, sample_images):
        annotated = pred.plot()  # BGR
        ax.imshow(annotated[..., ::-1])
        ax.set_title(src.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## Crop Detected Petri Dishes and Resize to 512x512

This section uses the trained detector (`best.pt`) to:
1. detect Petri dish bbox for each image,
2. crop by the highest-confidence box,
3. resize the crop to exactly `512x512`,
4. save all crops into a separate folder.


In [ ]:
# Crop inference configuration
CROP_SOURCE_DIR = DATASET_ROOT / "images" / "train"  # folder with source images
CROP_OUTPUT_DIR = DATASET_ROOT / "cropped_512"
CROP_REPORT_CSV = CROP_OUTPUT_DIR / "crop_report.csv"

TARGET_W = 720
TARGET_H = 720
CROP_CONF = 0.25
CROP_IOU = 0.5
OVERWRITE_CROPS = True

# Use best weights from training run if available, otherwise set manually.
CROP_MODEL_PATH = (
    run_dir / "weights" / "best.pt"
    if "run_dir" in globals()
    else Path("runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt")
)

print(f"Crop model: {Path(CROP_MODEL_PATH)}")
print(f"Source dir: {CROP_SOURCE_DIR}")
print(f"Output dir: {CROP_OUTPUT_DIR}")


In [ ]:
# Run detection-based cropping and resize to 512x512
import cv2

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

if not CROP_SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {CROP_SOURCE_DIR}")
if not Path(CROP_MODEL_PATH).exists():
    raise FileNotFoundError(f"Model weights not found: {CROP_MODEL_PATH}")

if OVERWRITE_CROPS and CROP_OUTPUT_DIR.exists():
    shutil.rmtree(CROP_OUTPUT_DIR)
CROP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

crop_model = YOLO(str(CROP_MODEL_PATH))
source_images = sorted([p for p in CROP_SOURCE_DIR.glob("*") if p.suffix.lower() in image_exts])

if not source_images:
    raise RuntimeError(f"No images found in: {CROP_SOURCE_DIR}")

rows = []
for i, img_path in enumerate(source_images, start=1):
    img = cv2.imread(str(img_path))
    if img is None:
        rows.append({"file": img_path.name, "status": "read_error"})
        continue

    h, w = img.shape[:2]
    pred = crop_model.predict(source=str(img_path), conf=CROP_CONF, iou=CROP_IOU, verbose=False)[0]

    if pred.boxes is None or len(pred.boxes) == 0:
        rows.append({"file": img_path.name, "status": "no_detection", "orig_w": w, "orig_h": h})
        continue

    confs = pred.boxes.conf.detach().cpu().numpy()
    boxes = pred.boxes.xyxy.detach().cpu().numpy()
    best_idx = int(np.argmax(confs))

    x1, y1, x2, y2 = boxes[best_idx]
    x1 = max(0, int(np.floor(x1)))
    y1 = max(0, int(np.floor(y1)))
    x2 = min(w, int(np.ceil(x2)))
    y2 = min(h, int(np.ceil(y2)))

    if x2 <= x1 or y2 <= y1:
        rows.append({"file": img_path.name, "status": "invalid_bbox", "orig_w": w, "orig_h": h})
        continue

    crop = img[y1:y2, x1:x2]
    resized = cv2.resize(crop, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

    out_path = CROP_OUTPUT_DIR / img_path.name
    cv2.imwrite(str(out_path), resized)

    rows.append(
        {
            "file": img_path.name,
            "status": "saved",
            "conf": float(confs[best_idx]),
            "orig_w": w,
            "orig_h": h,
            "bbox_w": x2 - x1,
            "bbox_h": y2 - y1,
            "out_w": TARGET_W,
            "out_h": TARGET_H,
            "out_path": str(out_path),
        }
    )

    if i % 25 == 0 or i == len(source_images):
        print(f"Processed {i}/{len(source_images)}")

report_df = pd.DataFrame(rows)
status_counts = report_df["status"].value_counts().rename_axis("status").reset_index(name="count")
display(status_counts)

saved_df = report_df[report_df["status"] == "saved"].copy()
report_df.to_csv(CROP_REPORT_CSV, index=False)
if not saved_df.empty:
    print(f"Saved crops: {len(saved_df)}")
    print(f"Crop output: {CROP_OUTPUT_DIR.resolve()}")
    print(f"Report CSV: {CROP_REPORT_CSV.resolve()}")
else:
    print("No crops were saved.")


In [ ]:
# Preview cropped images and confidence distribution
saved_df = report_df[report_df["status"] == "saved"].copy()

if saved_df.empty:
    print("No saved crops to display.")
else:
    preview_paths = [Path(p) for p in saved_df["out_path"].head(6).tolist()]

    n = len(preview_paths)
    cols = 3
    rows_n = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 4 * rows_n))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, p in zip(axes, preview_paths):
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(p.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    if "conf" in saved_df.columns:
        plt.figure(figsize=(6, 4))
        plt.hist(saved_df["conf"], bins=20, color="#3b82f6", edgecolor="black")
        plt.title("Confidence Distribution (Saved Crops)")
        plt.xlabel("Confidence")
        plt.ylabel("Count")
        plt.show()


In [ ]:
# Show no_detection files and preview
no_det_df = report_df[report_df["status"] == "no_detection"].copy()

if no_det_df.empty:
    print("No no_detection samples.")
else:
    display(no_det_df[["file", "orig_w", "orig_h"]].reset_index(drop=True))

    preview_files = no_det_df["file"].head(12).tolist()
    preview_paths = [CROP_SOURCE_DIR / f for f in preview_files if (CROP_SOURCE_DIR / f).exists()]

    if preview_paths:
        n = len(preview_paths)
        cols = 3
        rows_n = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 4 * rows_n))
        axes = np.array(axes).reshape(-1)

        for ax in axes:
            ax.axis("off")

        for ax, p in zip(axes, preview_paths):
            img = Image.open(p)
            ax.imshow(img)
            ax.set_title(p.name)
            ax.axis("off")

        plt.tight_layout()
        plt.show()


In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Predict on all train images without saving outputs to disk
weights = Path("runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt")
source_dir = Path("Новая папка") / "images" / "train"

if not weights.exists():
    raise FileNotFoundError(f"Weights not found: {weights}")
if not source_dir.exists():
    raise FileNotFoundError(f"Source directory not found: {source_dir}")

model = YOLO(str(weights))
processed = 0

for _ in model.predict(
    source=str(source_dir),
    save=False,
    save_txt=False,
    save_conf=False,
    save_crop=False,
    stream=True,
    verbose=True,
):
    processed += 1

print(f"Processed images: {processed}")


In [ ]:
from pathlib import Path
import math
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Predict + inline visualization (no files are written)
weights = Path("runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt")
source_dir = Path("Новая папка") / "images" / "train"

if not weights.exists():
    raise FileNotFoundError(f"Weights not found: {weights}")
if not source_dir.exists():
    raise FileNotFoundError(f"Source directory not found: {source_dir}")

model = YOLO(str(weights))
results = model.predict(
    source=str(source_dir),
    save=False,
    save_txt=False,
    save_conf=False,
    save_crop=False,
    stream=False,
    verbose=False,
)

print(f"Processed images: {len(results)}")

cols = 3
rows = max(1, math.ceil(len(results) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax, r in zip(axes, results):
    img = r.plot()[:, :, ::-1]  # BGR -> RGB
    ax.imshow(img)
    ax.set_title(Path(r.path).name, fontsize=9)
    ax.axis("off")

for ax in axes[len(results):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import shutil
import cv2
import numpy as np
from ultralytics import YOLO

# Crop images + adjust YOLO-seg labels, then resize to 720x720.
WEIGHTS = Path("runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt")
SRC_IMG_DIR = Path("Новая папка") / "images" / "train"
SRC_LBL_DIR = Path("Новая папка") / "labels" / "train"

# Safe default: write to a new folder. Set INPLACE=True only if you really want overwrite source files.
INPLACE = False
DST_ROOT = Path("Новая папка") / "cropped_720"
TARGET_W = 720
TARGET_H = 720
CONF = 0.25
IOU = 0.5
OVERWRITE_DST = True

if INPLACE:
    DST_IMG_DIR = SRC_IMG_DIR
    DST_LBL_DIR = SRC_LBL_DIR
else:
    DST_IMG_DIR = DST_ROOT / "images" / "train"
    DST_LBL_DIR = DST_ROOT / "labels" / "train"

if not WEIGHTS.exists():
    raise FileNotFoundError(f"Weights not found: {WEIGHTS}")
if not SRC_IMG_DIR.exists():
    raise FileNotFoundError(f"Source images dir not found: {SRC_IMG_DIR}")
if not SRC_LBL_DIR.exists():
    raise FileNotFoundError(f"Source labels dir not found: {SRC_LBL_DIR}")

if not INPLACE and OVERWRITE_DST and DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

DST_IMG_DIR.mkdir(parents=True, exist_ok=True)
DST_LBL_DIR.mkdir(parents=True, exist_ok=True)

def load_yolo_seg(txt_path: Path):
    anns = []
    if not txt_path.exists():
        return anns
    for raw in txt_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
            continue
        cls = parts[0]
        coords = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
        anns.append((cls, coords))
    return anns

def transform_seg(coords_norm, img_w, img_h, x1, y1, x2, y2):
    pts = coords_norm.copy()
    pts[:, 0] *= img_w
    pts[:, 1] *= img_h

    min_x, max_x = float(pts[:, 0].min()), float(pts[:, 0].max())
    min_y, max_y = float(pts[:, 1].min()), float(pts[:, 1].max())
    if max_x <= x1 or min_x >= x2 or max_y <= y1 or min_y >= y2:
        return None

    pts[:, 0] = np.clip(pts[:, 0], x1, x2 - 1e-6)
    pts[:, 1] = np.clip(pts[:, 1], y1, y2 - 1e-6)

    crop_w = max(1e-6, float(x2 - x1))
    crop_h = max(1e-6, float(y2 - y1))
    pts[:, 0] = (pts[:, 0] - x1) / crop_w
    pts[:, 1] = (pts[:, 1] - y1) / crop_h
    pts = np.clip(pts, 0.0, 1.0)

    if (pts[:, 0].max() - pts[:, 0].min()) < 1e-4:
        return None
    if (pts[:, 1].max() - pts[:, 1].min()) < 1e-4:
        return None
    return pts

def save_yolo_seg(txt_path: Path, anns):
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    with txt_path.open("w", encoding="utf-8") as f:
        for cls, pts in anns:
            flat = pts.reshape(-1)
            values = " ".join(f"{v:.6f}" for v in flat)
            f.write(f"{cls} {values}\\n")

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
images = sorted([p for p in SRC_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in image_exts])
if not images:
    raise RuntimeError(f"No images found in: {SRC_IMG_DIR}")

model = YOLO(str(WEIGHTS))

saved = 0
fallback_full = 0
dropped_masks = 0

for i, img_path in enumerate(images, start=1):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    h, w = img.shape[:2]
    pred = model.predict(source=str(img_path), conf=CONF, iou=IOU, verbose=False)[0]

    if pred.boxes is None or len(pred.boxes) == 0:
        x1, y1, x2, y2 = 0, 0, w, h
        fallback_full += 1
    else:
        confs = pred.boxes.conf.detach().cpu().numpy()
        boxes = pred.boxes.xyxy.detach().cpu().numpy()
        best_idx = int(np.argmax(confs))
        bx1, by1, bx2, by2 = boxes[best_idx]
        x1 = max(0, int(np.floor(bx1)))
        y1 = max(0, int(np.floor(by1)))
        x2 = min(w, int(np.ceil(bx2)))
        y2 = min(h, int(np.ceil(by2)))
        if x2 <= x1 or y2 <= y1:
            x1, y1, x2, y2 = 0, 0, w, h
            fallback_full += 1

    crop = img[y1:y2, x1:x2]
    interp = cv2.INTER_AREA if crop.shape[1] >= TARGET_W and crop.shape[0] >= TARGET_H else cv2.INTER_LINEAR
    resized = cv2.resize(crop, (TARGET_W, TARGET_H), interpolation=interp)

    out_img_path = DST_IMG_DIR / img_path.name
    cv2.imwrite(str(out_img_path), resized)

    in_lbl_path = SRC_LBL_DIR / f"{img_path.stem}.txt"
    out_lbl_path = DST_LBL_DIR / f"{img_path.stem}.txt"

    anns = load_yolo_seg(in_lbl_path)
    out_anns = []
    for cls, coords_norm in anns:
        new_pts = transform_seg(coords_norm, w, h, x1, y1, x2, y2)
        if new_pts is None:
            dropped_masks += 1
            continue
        out_anns.append((cls, new_pts))

    save_yolo_seg(out_lbl_path, out_anns)
    saved += 1

    if i % 25 == 0 or i == len(images):
        print(f"Processed {i}/{len(images)}")

print(f"Saved image/label pairs: {saved}")
print(f"Fallback to full-frame resize: {fallback_full}")
print(f"Dropped masks after crop: {dropped_masks}")
print(f"Images: {DST_IMG_DIR.resolve()}")
print(f"Labels: {DST_LBL_DIR.resolve()}")


In [ ]:
from pathlib import Path
import shutil
import cv2
import numpy as np
from ultralytics import YOLO

# Crop+resize dataset: Новая папка (2)/рч (images + PNG masks)
WEIGHTS = Path("runs/detect/runs/petri_curcle/yolo26n_petri_curcle/weights/best.pt")
SRC_ROOT = Path("Новая папка (2)") / "рч"
SRC_IMG_DIR = SRC_ROOT / "img"
SRC_MASK_HUMAN_DIR = SRC_ROOT / "masks_human"
SRC_MASK_MACHINE_DIR = SRC_ROOT / "masks_machine"
SRC_INST_DIR = SRC_ROOT / "masks_instances"

INPLACE = False
DST_ROOT = SRC_ROOT if INPLACE else SRC_ROOT / "cropped_720"
DST_IMG_DIR = DST_ROOT / "img"
DST_MASK_HUMAN_DIR = DST_ROOT / "masks_human"
DST_MASK_MACHINE_DIR = DST_ROOT / "masks_machine"
DST_INST_DIR = DST_ROOT / "masks_instances"

TARGET_W = 720
TARGET_H = 720
CONF = 0.25
IOU = 0.5
OVERWRITE_DST = True

if not WEIGHTS.exists():
    raise FileNotFoundError(f"Weights not found: {WEIGHTS}")
if not SRC_IMG_DIR.exists():
    raise FileNotFoundError(f"Images dir not found: {SRC_IMG_DIR}")
if not SRC_MASK_HUMAN_DIR.exists():
    raise FileNotFoundError(f"Mask dir not found: {SRC_MASK_HUMAN_DIR}")
if not SRC_MASK_MACHINE_DIR.exists():
    raise FileNotFoundError(f"Mask dir not found: {SRC_MASK_MACHINE_DIR}")
if not SRC_INST_DIR.exists():
    raise FileNotFoundError(f"Instances dir not found: {SRC_INST_DIR}")

if not INPLACE and OVERWRITE_DST and DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

DST_IMG_DIR.mkdir(parents=True, exist_ok=True)
DST_MASK_HUMAN_DIR.mkdir(parents=True, exist_ok=True)
DST_MASK_MACHINE_DIR.mkdir(parents=True, exist_ok=True)
DST_INST_DIR.mkdir(parents=True, exist_ok=True)

def crop_resize(arr, x1, y1, x2, y2, is_mask=False):
    crop = arr[y1:y2, x1:x2]
    if is_mask:
        interp = cv2.INTER_NEAREST
    else:
        interp = cv2.INTER_AREA if crop.shape[1] >= TARGET_W and crop.shape[0] >= TARGET_H else cv2.INTER_LINEAR
    return cv2.resize(crop, (TARGET_W, TARGET_H), interpolation=interp)

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
images = sorted([p for p in SRC_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in image_exts])
if not images:
    raise RuntimeError(f"No images found in: {SRC_IMG_DIR}")

model = YOLO(str(WEIGHTS))

saved_images = 0
saved_human = 0
saved_machine = 0
saved_instances = 0
fallback_full = 0
missing_human = 0
missing_machine = 0
missing_instances_dirs = 0

for i, img_path in enumerate(images, start=1):
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img is None:
        continue

    h, w = img.shape[:2]
    pred = model.predict(source=str(img_path), conf=CONF, iou=IOU, verbose=False)[0]

    if pred.boxes is None or len(pred.boxes) == 0:
        x1, y1, x2, y2 = 0, 0, w, h
        fallback_full += 1
    else:
        confs = pred.boxes.conf.detach().cpu().numpy()
        boxes = pred.boxes.xyxy.detach().cpu().numpy()
        best_idx = int(np.argmax(confs))
        bx1, by1, bx2, by2 = boxes[best_idx]
        x1 = max(0, int(np.floor(bx1)))
        y1 = max(0, int(np.floor(by1)))
        x2 = min(w, int(np.ceil(bx2)))
        y2 = min(h, int(np.ceil(by2)))
        if x2 <= x1 or y2 <= y1:
            x1, y1, x2, y2 = 0, 0, w, h
            fallback_full += 1

    out_img = crop_resize(img, x1, y1, x2, y2, is_mask=False)
    cv2.imwrite(str(DST_IMG_DIR / img_path.name), out_img)
    saved_images += 1

    in_human = SRC_MASK_HUMAN_DIR / f"{img_path.stem}.png"
    if in_human.exists():
        m = cv2.imread(str(in_human), cv2.IMREAD_UNCHANGED)
        if m is not None:
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            out_human = crop_resize(m, x1, y1, x2, y2, is_mask=True)
            cv2.imwrite(str(DST_MASK_HUMAN_DIR / in_human.name), out_human)
            saved_human += 1
    else:
        missing_human += 1

    in_machine = SRC_MASK_MACHINE_DIR / f"{img_path.stem}.png"
    if in_machine.exists():
        m = cv2.imread(str(in_machine), cv2.IMREAD_UNCHANGED)
        if m is not None:
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            out_machine = crop_resize(m, x1, y1, x2, y2, is_mask=True)
            cv2.imwrite(str(DST_MASK_MACHINE_DIR / in_machine.name), out_machine)
            saved_machine += 1
    else:
        missing_machine += 1

    in_inst_dir = SRC_INST_DIR / img_path.stem
    out_inst_dir = DST_INST_DIR / img_path.stem
    if in_inst_dir.exists() and in_inst_dir.is_dir():
        out_inst_dir.mkdir(parents=True, exist_ok=True)
        for inst_mask in sorted(in_inst_dir.glob("*.png")):
            m = cv2.imread(str(inst_mask), cv2.IMREAD_UNCHANGED)
            if m is None:
                continue
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            out_inst = crop_resize(m, x1, y1, x2, y2, is_mask=True)
            cv2.imwrite(str(out_inst_dir / inst_mask.name), out_inst)
            saved_instances += 1
    else:
        missing_instances_dirs += 1

    if i % 25 == 0 or i == len(images):
        print(f"Processed {i}/{len(images)}")

print(f"Saved images: {saved_images}")
print(f"Saved masks_human: {saved_human} (missing: {missing_human})")
print(f"Saved masks_machine: {saved_machine} (missing: {missing_machine})")
print(f"Saved instance masks: {saved_instances} (missing dirs: {missing_instances_dirs})")
print(f"Fallback to full-frame resize: {fallback_full}")
print(f"Output root: {DST_ROOT.resolve()}")
